# Grokking: Figure 1 reproduction

This notebook is a self-contained reproduction of the left panel of Figure 1 in Power et al., [Grokking: Generalization Beyond Overfitting on Small Algorithmic Datasets](https://arxiv.org/abs/2201.02177).

It trains a small decoder-only transformer on division modulo 97, using a fixed random 50/50 train-validation split. The paper-scale run uses Adam with no weight decay for 1,000,000 optimization steps and repeats the experiment with three model seeds. Training accuracy should approach 100% long before validation accuracy does.

Run the cells in order. The final training cell performs three million optimizer updates. Before it starts, a short device benchmark reports measured steps/second and a training-only runtime projection. Each seed is checkpointed every 10,000 steps and saved before the next seed is allocated; rerunning with the same experiment ID resumes the existing run.

In [1]:
from __future__ import annotations

import gc
import json
import math
import random
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

plt.style.use("seaborn-v0_8-whitegrid")
torch.set_float32_matmul_precision("high")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if getattr(torch.backends, "mps", None) is not None
    and torch.backends.mps.is_available()
    else "cpu"
)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass(frozen=True)
class Figure1Config:
    prime: int = 97
    train_fraction: float = 0.5
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 2
    mlp_mult: int = 4
    batch_size: int = 512
    learning_rate: float = 1e-3
    beta1: float = 0.9
    beta2: float = 0.98
    warmup_steps: int = 10
    max_steps: int = 1_000_000
    number_of_seeds: int = 3
    evaluation_points: int = 250
    checkpoint_interval: int = 10_000
    benchmark_warmup_steps: int = 10
    benchmark_steps: int = 50


CONFIG = Figure1Config()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")
CONFIG

PyTorch 2.14.0; device = mps


/Users/ashwin/Code/mech_interp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Figure1Config(prime=97, train_fraction=0.5, d_model=128, n_heads=4, n_layers=2, mlp_mult=4, batch_size=512, learning_rate=0.001, beta1=0.9, beta2=0.98, warmup_steps=10, max_steps=1000000, number_of_seeds=3, evaluation_points=250, checkpoint_interval=10000, benchmark_warmup_steps=10, benchmark_steps=50)

## Dataset and tokenization

There are 97 choices for the numerator and 96 nonzero choices for the denominator, giving 9,312 equations. Division is multiplication by the denominator's modular inverse.

Following the authors' released implementation, an equation is tokenized as

    <eos> x / y = answer

and teacher-forced targets are

    answer <eos>

Loss and equation-level accuracy are calculated only on those two right-hand-side targets. The vocabulary is shared with the paper's other arithmetic and permutation tasks; retaining that output vocabulary avoids making this task artificially easier. A fixed NumPy seed shuffles the equations once, after which the first half is training data and the second half is validation data.

In [2]:
PAPER_OPERATORS = sorted(
    [
        "+", "-", "*", "/", "**2+", "**3+", "+*", "+-",
        "(x._value//y)if(y._value%2==1)else(x-y)_mod_97",
        "copy", "reverse", "s5", "s5aba", "s5conj", "sort",
        "x**2+y**2_mod_97", "x**2+y**2+x*y_mod_97",
        "x**2+y**2+x*y+x_mod_97", "x**3+x*y_mod_97",
        "x**3+x*y**2+y_mod_97",
    ]
)

EOS_TOKEN = 0
EQUALS_TOKEN = 1
OPERATOR_TOKEN = 2 + PAPER_OPERATORS.index("/")
NUMBER_OFFSET = 2 + len(PAPER_OPERATORS)
VOCAB_SIZE = NUMBER_OFFSET + CONFIG.prime + math.factorial(5)


def division_mod_prime(x: int, y: int, prime: int) -> int:
    if y == 0:
        raise ValueError("Zero has no multiplicative inverse.")
    return (x * pow(y, -1, prime)) % prime


def build_figure1_datasets(
    config: Figure1Config,
) -> tuple[TensorDataset, TensorDataset]:
    equations: list[list[int]] = []
    targets: list[list[int]] = []

    # Match the released dataset's pre-shuffle order: answer first,
    # denominator second, then derive the numerator = denominator * answer.
    for answer in range(config.prime):
        for denominator in range(1, config.prime):
            numerator = (denominator * answer) % config.prime
            assert division_mod_prime(numerator, denominator, config.prime) == answer

            answer_token = NUMBER_OFFSET + answer
            equations.append(
                [
                    EOS_TOKEN,
                    NUMBER_OFFSET + numerator,
                    OPERATOR_TOKEN,
                    NUMBER_OFFSET + denominator,
                    EQUALS_TOKEN,
                    answer_token,
                ]
            )
            targets.append([answer_token, EOS_TOKEN])

    permutation = np.random.RandomState(0).permutation(len(equations))
    inputs = torch.tensor(np.asarray(equations)[permutation], dtype=torch.long)
    labels = torch.tensor(np.asarray(targets)[permutation], dtype=torch.long)

    split_index = round(config.train_fraction * len(inputs))
    return (
        TensorDataset(inputs[:split_index], labels[:split_index]),
        TensorDataset(inputs[split_index:], labels[split_index:]),
    )


def decode_equation(tokens: torch.Tensor) -> str:
    x = int(tokens[1]) - NUMBER_OFFSET
    y = int(tokens[3]) - NUMBER_OFFSET
    answer = int(tokens[5]) - NUMBER_OFFSET
    return f"{x} / {y} = {answer} (mod {CONFIG.prime})"


TRAIN_DATASET, VAL_DATASET = build_figure1_datasets(CONFIG)

assert len(TRAIN_DATASET) == len(VAL_DATASET) == 4_656
assert not set(map(tuple, TRAIN_DATASET.tensors[0].tolist())) & set(
    map(tuple, VAL_DATASET.tensors[0].tolist())
)

print(f"vocabulary size: {VOCAB_SIZE}")
print(f"training equations: {len(TRAIN_DATASET):,}")
print(f"validation equations: {len(VAL_DATASET):,}")
for index in range(5):
    print(" ", decode_equation(TRAIN_DATASET[index][0]))

vocabulary size: 239
training equations: 4,656
validation equations: 4,656
  77 / 47 = 78 (mod 97)
  11 / 23 = 30 (mod 97)
  18 / 3 = 6 (mod 97)
  62 / 82 = 67 (mod 97)
  92 / 25 = 58 (mod 97)


## Model

The network is a causal, decoder-only transformer with two post-normalized blocks, width 128, four attention heads, a 4× ReLU MLP, sinusoidal position encodings, and no dropout. It reads the six-token teacher-forced equation and emits predictions after the equals sign and after the answer token.

For throughput, Q/K/V are each projected for all heads in one matrix multiplication, and attention uses PyTorch's fused scaled-dot-product attention. This is mathematically equivalent to the released per-head implementation, but floating-point operation order and backend kernels make it not bitwise identical.

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int) -> None:
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")

        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length, _ = hidden.shape
        qkv = self.qkv(hidden).view(
            batch_size, sequence_length, 3, self.n_heads, self.d_head
        )
        query, key, value = qkv.unbind(dim=2)
        query = query.transpose(1, 2)
        key = key.transpose(1, 2)
        value = value.transpose(1, 2)

        attended = F.scaled_dot_product_attention(
            query,
            key,
            value,
            dropout_p=0.0,
            is_causal=True,
        )
        attended = attended.transpose(1, 2).contiguous().view(
            batch_size, sequence_length, -1
        )
        return self.output(attended)


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_mult: int) -> None:
        super().__init__()
        self.attention = CausalSelfAttention(d_model, n_heads)
        self.attention_norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_mult * d_model, bias=False),
            nn.ReLU(),
            nn.Linear(mlp_mult * d_model, d_model, bias=False),
        )
        self.mlp_norm = nn.LayerNorm(d_model)

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        hidden = self.attention_norm(hidden + self.attention(hidden))
        return self.mlp_norm(hidden + self.mlp(hidden))


class GrokkingTransformer(nn.Module):
    def __init__(self, config: Figure1Config) -> None:
        super().__init__()
        if config.d_model % 2:
            raise ValueError("Sinusoidal positions require an even d_model")

        self.embedding = nn.Embedding(VOCAB_SIZE, config.d_model)

        positions = torch.arange(6, dtype=torch.float32).unsqueeze(1)
        dimensions = torch.arange(0, config.d_model, 2, dtype=torch.float32)
        angles = positions / (10_000 ** (dimensions / config.d_model))
        position_encoding = torch.empty(6, config.d_model)
        position_encoding[:, 0::2] = torch.sin(angles)
        position_encoding[:, 1::2] = torch.cos(angles)
        self.register_buffer("position_encoding", position_encoding, persistent=False)

        self.blocks = nn.ModuleList(
            DecoderBlock(config.d_model, config.n_heads, config.mlp_mult)
            for _ in range(config.n_layers)
        )
        self.unembedding = nn.Linear(config.d_model, VOCAB_SIZE, bias=False)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        sequence_length = tokens.size(1)
        hidden = self.embedding(tokens)
        hidden = hidden + self.position_encoding[:sequence_length]

        for block in self.blocks:
            hidden = block(hidden)

        # Position -2 predicts the answer; position -1 predicts trailing EOS.
        return self.unembedding(hidden[:, -2:, :])


preview_model = GrokkingTransformer(CONFIG)
preview_logits = preview_model(TRAIN_DATASET.tensors[0][:4])
print(f"logit shape: {tuple(preview_logits.shape)}")
print(f"parameters: {sum(p.numel() for p in preview_model.parameters()):,}")
del preview_model, preview_logits

logit shape: (4, 2, 239)
parameters: 455,424


## Training, checkpointing, and evaluation

Each reported accuracy is equation-level: both the answer and trailing end token must be correct. Evaluation occurs at logarithmically spaced steps so the plot resolves both early memorization and late generalization without repeatedly evaluating at all one million updates.

Each seed has its own checkpoint and final-result file. A checkpoint contains the model, optimizer, metric history, Python/NumPy/PyTorch RNG states, and the shuffled batch order plus next-batch position. Writes use a temporary file followed by an atomic rename. Reusing an experiment ID resumes compatible checkpoints; delete that experiment folder to start fresh.

In [4]:
@torch.inference_mode()
def evaluate(
    model: nn.Module,
    dataset: TensorDataset,
    device: torch.device,
) -> dict[str, float]:
    model.eval()
    loader = DataLoader(dataset, batch_size=min(2048, len(dataset)), shuffle=False)
    loss_function = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total_correct = 0
    total_equations = 0

    for tokens, targets in loader:
        tokens, targets = tokens.to(device), targets.to(device)
        logits = model(tokens)
        total_loss += float(
            loss_function(logits.flatten(0, 1), targets.flatten()).item()
        )
        predictions = logits.argmax(dim=-1)
        total_correct += int((predictions == targets).all(dim=-1).sum().item())
        total_equations += targets.size(0)

    return {
        "loss": total_loss / (2 * total_equations),
        "accuracy": total_correct / total_equations,
    }


def make_evaluation_steps(max_steps: int, number_of_points: int) -> set[int]:
    steps = np.geomspace(1, max_steps, num=number_of_points)
    return set(np.unique(np.rint(steps).astype(int)).tolist()) | {1, max_steps}


def first_step_at_accuracy(
    history: list[dict[str, float]],
    split: str,
    threshold: float = 0.99,
) -> int | None:
    for row in history:
        if row[f"{split}_accuracy"] >= threshold:
            return int(row["step"])
    return None


class ShuffledBatchStream:
    """A resumable, single-process shuffled batch stream."""

    def __init__(
        self,
        dataset: TensorDataset,
        batch_size: int,
        seed: int,
    ) -> None:
        self.inputs, self.targets = dataset.tensors
        self.batch_size = min(batch_size, math.ceil(len(dataset) / 2))
        self.generator = torch.Generator().manual_seed(seed)
        self.order = torch.randperm(len(dataset), generator=self.generator)
        self.position = 0
        self.batch_in_epoch = 0
        self.epoch = 0

    def next(self) -> tuple[torch.Tensor, torch.Tensor]:
        if self.position >= len(self.order):
            self.order = torch.randperm(
                len(self.order), generator=self.generator
            )
            self.position = 0
            self.batch_in_epoch = 0
            self.epoch += 1

        end = min(self.position + self.batch_size, len(self.order))
        indices = self.order[self.position:end]
        self.position = end
        self.batch_in_epoch += 1
        return self.inputs[indices], self.targets[indices]

    def state_dict(self) -> dict[str, object]:
        return {
            "generator_state": self.generator.get_state(),
            "order": self.order,
            "position": self.position,
            "batch_in_epoch": self.batch_in_epoch,
            "epoch": self.epoch,
        }

    def load_state_dict(self, state: dict[str, object]) -> None:
        order = state["order"].cpu()
        if len(order) != len(self.inputs):
            raise ValueError("Checkpoint batch order does not match the dataset.")
        self.generator.set_state(state["generator_state"].cpu())
        self.order = order
        self.position = int(state["position"])
        self.batch_in_epoch = int(state["batch_in_epoch"])
        self.epoch = int(state["epoch"])


def capture_rng_state() -> dict[str, object]:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": (
            [state.cpu() for state in torch.cuda.get_rng_state_all()]
            if torch.cuda.is_available()
            else None
        ),
    }


def restore_rng_state(state: dict[str, object]) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"].cpu())
    if torch.cuda.is_available() and state["torch_cuda"] is not None:
        torch.cuda.set_rng_state_all(
            [cuda_state.cpu() for cuda_state in state["torch_cuda"]]
        )


def atomic_torch_save(payload: dict[str, object], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary_path)
    temporary_path.replace(path)


def seed_directory(run_directory: Path, model_seed: int) -> Path:
    return run_directory / f"seed_{model_seed}"


def checkpoint_path(run_directory: Path, model_seed: int) -> Path:
    return seed_directory(run_directory, model_seed) / "checkpoint.pt"


def result_path(run_directory: Path, model_seed: int) -> Path:
    return seed_directory(run_directory, model_seed) / "result.pt"


def validate_saved_config(
    saved_config: dict[str, object],
    config: Figure1Config,
    path: Path,
) -> None:
    if saved_config != asdict(config):
        raise ValueError(
            f"{path} was created with a different configuration. "
            "Use a new experiment ID or delete the existing experiment folder."
        )


def save_checkpoint(
    path: Path,
    config: Figure1Config,
    model_seed: int,
    step: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    history: list[dict[str, float]],
    batch_stream: ShuffledBatchStream,
) -> None:
    atomic_torch_save(
        {
            "format_version": 1,
            "config": asdict(config),
            "model_seed": model_seed,
            "step": step,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "history": history,
            "rng_state": capture_rng_state(),
            "batch_stream_state": batch_stream.state_dict(),
        },
        path,
    )


def save_seed_result(
    path: Path,
    config: Figure1Config,
    model_seed: int,
    model: nn.Module,
    history: list[dict[str, float]],
) -> None:
    cpu_model_state = {
        name: tensor.detach().cpu() for name, tensor in model.state_dict().items()
    }
    atomic_torch_save(
        {
            "format_version": 1,
            "config": asdict(config),
            "model_seed": model_seed,
            "step": config.max_steps,
            "model_state": cpu_model_state,
            "history": history,
        },
        path,
    )


def load_seed_history(
    path: Path,
    config: Figure1Config,
) -> list[dict[str, float]]:
    result = torch.load(path, map_location="cpu", weights_only=False)
    validate_saved_config(result["config"], config, path)
    return result["history"]


def completed_steps(
    run_directory: Path,
    config: Figure1Config,
    model_seed: int,
) -> int:
    final_path = result_path(run_directory, model_seed)
    if final_path.exists():
        result = torch.load(final_path, map_location="cpu", weights_only=False)
        validate_saved_config(result["config"], config, final_path)
        return config.max_steps

    resume_path = checkpoint_path(run_directory, model_seed)
    if resume_path.exists():
        checkpoint = torch.load(
            resume_path, map_location="cpu", weights_only=False
        )
        validate_saved_config(checkpoint["config"], config, resume_path)
        return int(checkpoint["step"])

    return 0


def synchronize_device() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


def benchmark_steps_per_second(config: Figure1Config) -> float:
    seed_everything(1_000_003)
    model = GrokkingTransformer(config).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        betas=(config.beta1, config.beta2),
        weight_decay=0.0,
    )
    loss_function = nn.CrossEntropyLoss()
    batch_size = min(config.batch_size, math.ceil(len(TRAIN_DATASET) / 2))
    tokens = TRAIN_DATASET.tensors[0][:batch_size].to(DEVICE)
    targets = TRAIN_DATASET.tensors[1][:batch_size].to(DEVICE)

    def benchmark_step() -> None:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(tokens)
        loss = loss_function(logits.flatten(0, 1), targets.flatten())
        loss.backward()
        optimizer.step()

    for _ in range(config.benchmark_warmup_steps):
        benchmark_step()
    synchronize_device()

    started_at = time.perf_counter()
    for _ in range(config.benchmark_steps):
        benchmark_step()
    synchronize_device()
    elapsed = time.perf_counter() - started_at

    del model, optimizer, tokens, targets
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return config.benchmark_steps / elapsed


def format_duration(seconds: float) -> str:
    hours, remainder = divmod(int(round(seconds)), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:d}h {minutes:02d}m {seconds:02d}s"


def train_one_seed(
    config: Figure1Config,
    model_seed: int,
    run_directory: Path,
) -> list[dict[str, float]]:
    final_path = result_path(run_directory, model_seed)
    if final_path.exists():
        print(f"seed {model_seed}: loading completed result from {final_path}")
        return load_seed_history(final_path, config)

    seed_everything(model_seed)
    batch_stream = ShuffledBatchStream(
        TRAIN_DATASET, config.batch_size, model_seed
    )
    model = GrokkingTransformer(config).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        betas=(config.beta1, config.beta2),
        weight_decay=0.0,
    )
    loss_function = nn.CrossEntropyLoss()
    history: list[dict[str, float]] = []
    first_step = 1

    resume_path = checkpoint_path(run_directory, model_seed)
    if resume_path.exists():
        checkpoint = torch.load(
            resume_path, map_location=DEVICE, weights_only=False
        )
        validate_saved_config(checkpoint["config"], config, resume_path)
        if int(checkpoint["model_seed"]) != model_seed:
            raise ValueError(f"Seed mismatch in {resume_path}.")
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        history = checkpoint["history"]
        batch_stream.load_state_dict(checkpoint["batch_stream_state"])
        restore_rng_state(checkpoint["rng_state"])
        first_step = int(checkpoint["step"]) + 1
        print(
            f"seed {model_seed}: resuming at step {first_step:,} "
            f"(epoch {batch_stream.epoch}, batch {batch_stream.batch_in_epoch})"
        )

    evaluation_steps = make_evaluation_steps(
        config.max_steps, config.evaluation_points
    )
    progress = tqdm(
        range(first_step, config.max_steps + 1),
        desc=f"seed {model_seed}",
        unit="step",
        initial=first_step - 1,
        total=config.max_steps,
        mininterval=1.0,
    )

    for step in progress:
        tokens, targets = batch_stream.next()

        # Linear warmup over the first ten updates, then a constant 1e-3.
        learning_rate = config.learning_rate * min(
            step / config.warmup_steps, 1.0
        )
        for parameter_group in optimizer.param_groups:
            parameter_group["lr"] = learning_rate

        model.train()
        tokens, targets = tokens.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(tokens)
        loss = loss_function(logits.flatten(0, 1), targets.flatten())
        loss.backward()
        optimizer.step()

        if step in evaluation_steps:
            train_metrics = evaluate(model, TRAIN_DATASET, DEVICE)
            validation_metrics = evaluate(model, VAL_DATASET, DEVICE)
            row = {
                "step": step,
                "train_loss": train_metrics["loss"],
                "train_accuracy": train_metrics["accuracy"],
                "validation_loss": validation_metrics["loss"],
                "validation_accuracy": validation_metrics["accuracy"],
            }
            history.append(row)
            progress.set_postfix(
                train=f"{row['train_accuracy']:.3f}",
                validation=f"{row['validation_accuracy']:.3f}",
            )

        if step % config.checkpoint_interval == 0 or step == config.max_steps:
            save_checkpoint(
                resume_path,
                config,
                model_seed,
                step,
                model,
                optimizer,
                history,
                batch_stream,
            )

    save_seed_result(final_path, config, model_seed, model, history)
    print(f"seed {model_seed}: saved result to {final_path}")
    return history


def summarize(history: list[dict[str, float]], model_seed: int) -> None:
    final = history[-1]
    print(
        f"seed {model_seed}: "
        f"train={final['train_accuracy']:.3%}, "
        f"validation={final['validation_accuracy']:.3%}, "
        f"train>=99% at {first_step_at_accuracy(history, 'train')}, "
        f"validation>=99% at {first_step_at_accuracy(history, 'validation')}"
    )

In [5]:
# This identifier names the persistent folder. Reuse it to resume; delete the
# folder (or choose a new ID) for a fresh run.
EXPERIMENT_ID = "figure1_division_p97_sdpa_v1"

# This is the paper-scale experiment: 3 seeds x 1,000,000 updates.
# For a quick plumbing check, use a different EXPERIMENT_ID and try:
# run_config = replace(CONFIG, max_steps=20, number_of_seeds=1,
#                      evaluation_points=10, checkpoint_interval=10)
run_config = CONFIG


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not find the repository root from the current directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_DIRECTORY = PROJECT_ROOT / "artifacts" / "grokking" / EXPERIMENT_ID
MANIFEST_PATH = RUN_DIRECTORY / "manifest.json"

if RUN_DIRECTORY.exists():
    print(f"Found experiment folder; resuming from {RUN_DIRECTORY}")
else:
    RUN_DIRECTORY.mkdir(parents=True)
    print(f"Created experiment folder at {RUN_DIRECTORY}")

manifest = {
    "experiment_id": EXPERIMENT_ID,
    "config": asdict(run_config),
    "attention": "combined_qkv_scaled_dot_product_attention",
}
if MANIFEST_PATH.exists():
    saved_manifest = json.loads(MANIFEST_PATH.read_text())
    if saved_manifest != manifest:
        raise ValueError(
            "The experiment folder's manifest does not match this run. "
            "Change EXPERIMENT_ID or delete the existing folder."
        )
else:
    temporary_manifest = MANIFEST_PATH.with_suffix(".json.tmp")
    temporary_manifest.write_text(json.dumps(manifest, indent=2) + "\n")
    temporary_manifest.replace(MANIFEST_PATH)

remaining_steps = sum(
    run_config.max_steps
    - completed_steps(RUN_DIRECTORY, run_config, model_seed)
    for model_seed in range(run_config.number_of_seeds)
)

if remaining_steps:
    steps_per_second = benchmark_steps_per_second(run_config)
    projected_seconds = remaining_steps / steps_per_second
    print(f"benchmark: {steps_per_second:.2f} optimizer steps/second")
    print(
        "projected remaining training time: "
        f"{format_duration(projected_seconds)} "
        "(evaluation and checkpoint I/O add extra time)"
    )
else:
    print("All seeds already have completed results; no training is needed.")

histories: list[list[dict[str, float]]] = []
started_at = time.perf_counter()

for model_seed in range(run_config.number_of_seeds):
    history = train_one_seed(run_config, model_seed, RUN_DIRECTORY)
    histories.append(history)
    summarize(history, model_seed)

    # train_one_seed returns metrics only, so no trained model survives here.
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()

print(f"elapsed this session: {(time.perf_counter() - started_at) / 3600:.2f} hours")

Found experiment folder; resuming from /Users/ashwin/Code/mech_interp/artifacts/grokking/figure1_division_p97_sdpa_v1
benchmark: 33.14 optimizer steps/second
projected remaining training time: 24h 53m 41s (evaluation and checkpoint I/O add extra time)
seed 0: resuming at step 30,001 (epoch 2999, batch 10)


seed 0:   3%|▏    | 30386/1000000 [00:17<12:17:57, 21.90step/s, train=1.000, validation=0.020]


KeyboardInterrupt: 

In [ ]:
def plot_figure1_left(
    histories: list[list[dict[str, float]]],
) -> tuple[plt.Figure, plt.Axes]:
    figure, axis = plt.subplots(figsize=(6.4, 4.1))

    for seed_index, history in enumerate(histories):
        steps = [row["step"] for row in history]
        axis.plot(
            steps,
            [100 * row["train_accuracy"] for row in history],
            color="#ff0000",
            alpha=0.7,
            linewidth=1.25,
            label="training" if seed_index == 0 else None,
        )
        axis.plot(
            steps,
            [100 * row["validation_accuracy"] for row in history],
            color="#008000",
            alpha=0.7,
            linewidth=1.25,
            label="validation" if seed_index == 0 else None,
        )

    axis.set(
        xscale="log",
        xlim=(1, run_config.max_steps),
        ylim=(-2, 102),
        title="Modular division (training on 50% of data)",
        xlabel="Optimization steps",
        ylabel="Accuracy (%)",
    )
    axis.legend()
    figure.tight_layout()
    return figure, axis


# histories contains only compact metric records; trained models live on disk.
figure, axis = plot_figure1_left(histories)
plt.show()